# Phase 1 — Causal Circuit Interventions

Tests whether discovered CTLS circuits are functionally meaningful via norm-bounded input-space interventions.

| Section | Content |
|---------|---------|
| 3. Fit Evaluation Probe | Frozen-feature linear probe for a task-level readout |
| 4. Select Circuit Set | Mixed class-specific and class-agnostic circuits |
| 5. Quantitative Causal Interventions | Activation / suppression + matched controls |
| 6. Qualitative Counterfactual Examples | Original / intervened / difference maps |
| 7. Summary Table | Aggregate causal effects by circuit type and control |

**Prerequisites:** a trained checkpoint from `nb01_training_and_validation.ipynb`.
This notebook uses an evaluation-only linear probe rather than the untrained backbone head.

## 0. Colab Setup

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/jacobposchl/trainable-circuits'
REPO_DIR = '/content/model_interpretability'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'], check=True)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data'
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print(f'Repo:     {REPO_DIR}')
print(f'Data:     {DATA_DIR}')
print(f'Checkpts: {CKPT_DIR}')


## 1. Configuration

Edit **only this cell** to switch the model and the intervention budget.

In [ ]:
# -----------------------------------------------------------------------------
#  SELECT MODEL  —  uncomment one line
# -----------------------------------------------------------------------------
MODEL_CONFIG = CONFIG_DIR + '/models/resnet18.yaml'
# MODEL_CONFIG = CONFIG_DIR + '/models/resnet34.yaml'
# MODEL_CONFIG = CONFIG_DIR + '/models/resnet50.yaml'

# Probe + data settings
DISCOVERY_SAMPLES     = 1200
INTERVENTION_SAMPLES  = 256
PROBE_MAX_TRAIN       = 4000
PROBE_MAX_VAL         = 1500
PROBE_EPOCHS          = 80

# Circuit-selection defaults
N_SPECIFIC            = 2
N_AGNOSTIC            = 2
MIN_CIRCUIT_SIZE      = 20

# PGD intervention defaults (pixel-space L8 budget)
EPS_PIXELS            = 8.0 / 255.0
STEP_PIXELS           = 2.0 / 255.0
N_STEPS               = 8
INTERVENTION_BATCH    = 32
QUAL_EXAMPLES         = 4

print(f'Using config: {MODEL_CONFIG}')


## 2. Load Model, Data, and Discover/Load Circuits

In [ ]:
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from evaluation import (CircuitAnalyzer, SpanCentricDiscovery, build_circuit_library, build_control_prototypes, fit_linear_probe, load_checkpoint, run_intervention_batch, select_circuit_set, summarize_intervention_results)
from evaluation import within_span_elevation
from data.cifar import get_standard_loaders

with open(MODEL_CONFIG) as f:
    config = yaml.safe_load(f)

config['data']['data_dir']          = DATA_DIR
config['logging']['checkpoint_dir'] = CKPT_DIR + '/' + config['experiment']['name']

checkpoint_path = config['logging']['checkpoint_dir'] + '/best.pt'
device          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

backbone, meta_encoder, info_loss = load_checkpoint(config, checkpoint_path, device)

train_loader, val_loader = get_standard_loaders(data_dir=config['data']['data_dir'], batch_size=config['data'].get('batch_size', 256), num_workers=config['data'].get('num_workers', 4), augment=False)


In [ ]:
TOTAL_SAMPLES = DISCOVERY_SAMPLES + INTERVENTION_SAMPLES

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
print(f'Collecting representations (max {TOTAL_SAMPLES} samples)...')
data = analyzer.collect_representations(max_samples=TOTAL_SAMPLES)

z_all      = data['z_list']
images_all = data['images']
labels_all = data['labels']
L          = len(z_all)

discovery_slice    = slice(0, DISCOVERY_SAMPLES)
intervention_slice = slice(DISCOVERY_SAMPLES, DISCOVERY_SAMPLES + INTERVENTION_SAMPLES)

z_discovery      = [z[discovery_slice].numpy() for z in z_all]
images_holdout   = images_all[intervention_slice].to(device)
labels_discovery = labels_all[discovery_slice]
labels_holdout   = labels_all[intervention_slice]

print(f'Discovery samples : {labels_discovery.shape[0]}')
print(f'Holdout samples   : {labels_holdout.shape[0]}')


In [ ]:
disc_cfg  = config.get('discovery', {})
discovery = SpanCentricDiscovery(n_layers=L, umap_n_components=disc_cfg.get('umap_n_components', 15), umap_n_neighbors=disc_cfg.get('umap_n_neighbors', 15), min_cluster_fraction=disc_cfg.get('min_cluster_fraction', 0.01), max_cluster_fraction=disc_cfg.get('max_cluster_fraction', 0.40), min_cluster_size=disc_cfg.get('hdbscan_min_cluster_size', 5))

print(f'Running discovery across {L*(L+1)//2} candidate spans...')
circuits = discovery.discover_all(z_discovery)
population_sims = discovery.compute_span_similarities(z_discovery, span=(0, L - 1))

for circuit in circuits:
    cluster_sims = discovery.compute_span_similarities(z_discovery, span=circuit['span'], image_mask=circuit['image_mask'])
    circuit.update(within_span_elevation(cluster_sims, population_sims))

library = build_circuit_library(circuits, z_discovery, labels_discovery)
print(f'Discovered {len(library)} annotated circuits.')


## 3. Fit Evaluation Probe

In [ ]:
probe, probe_metrics = fit_linear_probe(backbone, train_loader, val_loader, device, max_train_samples=PROBE_MAX_TRAIN, max_val_samples=PROBE_MAX_VAL, epochs=PROBE_EPOCHS)

print(f"Linear probe best val acc: {probe_metrics['best_val_acc']:.4f}")


## 4. Select Circuit Set

In [ ]:
selected = select_circuit_set(library, n_specific=N_SPECIFIC, n_agnostic=N_AGNOSTIC, min_size=MIN_CIRCUIT_SIZE)

selected_df = pd.DataFrame([{
    'name': c['name'],
    'type': c['circuit_type'],
    'span': f"L{c['span'][0]+1}-L{c['span'][1]+1}",
    'size': c['size'],
    'purity': c['purity'],
    'elevation_sigma': c.get('elevation_sigma', float('nan')),
    'associated_label': c.get('associated_label'),
} for c in selected])
selected_df


In [ ]:
control_map = {c['name']: build_control_prototypes(c, library, z_discovery, seed=idx) for idx, c in enumerate(selected)}

print('Controls prepared for:')
for name, controls in control_map.items():
    print(f'  {name}: {list(controls)}')


## 5. Quantitative Causal Interventions

In [ ]:
images_batch = images_holdout[:INTERVENTION_BATCH]
results = []

for circuit in selected:
    targets = {'target': circuit['prototype']}
    targets.update(control_map[circuit['name']])

    for target_name, prototype in targets.items():
        for mode in ('activate', 'suppress'):
            out = run_intervention_batch(backbone, meta_encoder, probe, images_batch, prototype, mode=mode, eps_pixels=EPS_PIXELS, step_pixels=STEP_PIXELS, n_steps=N_STEPS)
            out['summary']['selected_circuit'] = circuit['name']
            out['summary']['target_kind']      = target_name
            results.append(out)

summary_rows = summarize_intervention_results(results)
summary_df   = pd.DataFrame(summary_rows)
summary_df


In [ ]:
group_cols = ['selected_circuit', 'circuit_type', 'target_kind', 'mode']
metric_cols = ['delta_score', 'delta_confidence', 'delta_entropy', 'delta_associated_logit', 'delta_associated_prob', 'delta_associated_pred_rate']
present_metrics = [c for c in metric_cols if c in summary_df.columns]
agg_df = summary_df[group_cols + present_metrics].groupby(group_cols, dropna=False).mean().reset_index()
agg_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_only = summary_df[summary_df['target_kind'] == 'target']

for mode, color in [('activate', 'steelblue'), ('suppress', 'coral')]:
    subset = target_only[target_only['mode'] == mode]
    axes[0].bar(subset['selected_circuit'] + ' | ' + mode, subset['delta_score'], color=color, alpha=0.8)
axes[0].set_title('Target Circuit Score Shift')
axes[0].tick_params(axis='x', rotation=90)
axes[0].axhline(0, color='black', linewidth=1)

plot_metric = 'delta_associated_logit' if 'delta_associated_logit' in summary_df.columns else 'delta_entropy'
for mode, color in [('activate', 'seagreen'), ('suppress', 'goldenrod')]:
    subset = target_only[target_only['mode'] == mode]
    axes[1].bar(subset['selected_circuit'] + ' | ' + mode, subset[plot_metric].fillna(0.0), color=color, alpha=0.8)
axes[1].set_title(f'{plot_metric} (target only)')
axes[1].tick_params(axis='x', rotation=90)
axes[1].axhline(0, color='black', linewidth=1)
fig.tight_layout()
plt.show()


## 6. Qualitative Counterfactual Examples

In [ ]:
from evaluation.circuit_analysis import denormalize

def show_counterfactual_triplets(original, intervened, title, n_show=QUAL_EXAMPLES):
    diff = (denormalize(intervened) - denormalize(original)).abs()
    n_show = min(n_show, original.shape[0])
    fig, axes = plt.subplots(n_show, 3, figsize=(8, 2.5 * n_show))
    if n_show == 1:
        axes = np.array([axes])
    for i in range(n_show):
        axes[i, 0].imshow(denormalize(original[i]).cpu().permute(1, 2, 0).numpy())
        axes[i, 0].set_title('Original')
        axes[i, 1].imshow(denormalize(intervened[i]).cpu().permute(1, 2, 0).numpy())
        axes[i, 1].set_title('Intervened')
        axes[i, 2].imshow(diff[i].cpu().permute(1, 2, 0).numpy())
        axes[i, 2].set_title('|?|')
        for ax in axes[i]:
            ax.axis('off')
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


In [ ]:
specific = next((c for c in selected if c['circuit_type'] == 'class_specific'), None)
agnostic = next((c for c in selected if c['circuit_type'] == 'class_agnostic'), None)

qual_batch = images_holdout[:QUAL_EXAMPLES]

if specific is not None:
    activated = run_intervention_batch(backbone, meta_encoder, probe, qual_batch, specific['prototype'], mode='activate', eps_pixels=EPS_PIXELS, step_pixels=STEP_PIXELS, n_steps=N_STEPS)
    suppressed = run_intervention_batch(backbone, meta_encoder, probe, qual_batch, specific['prototype'], mode='suppress', eps_pixels=EPS_PIXELS, step_pixels=STEP_PIXELS, n_steps=N_STEPS)
    show_counterfactual_triplets(qual_batch.cpu(), activated['images_after'], f"Specific circuit activation: {specific['name']}")
    show_counterfactual_triplets(qual_batch.cpu(), suppressed['images_after'], f"Specific circuit suppression: {specific['name']}")

if agnostic is not None:
    activated = run_intervention_batch(backbone, meta_encoder, probe, qual_batch, agnostic['prototype'], mode='activate', eps_pixels=EPS_PIXELS, step_pixels=STEP_PIXELS, n_steps=N_STEPS)
    suppressed = run_intervention_batch(backbone, meta_encoder, probe, qual_batch, agnostic['prototype'], mode='suppress', eps_pixels=EPS_PIXELS, step_pixels=STEP_PIXELS, n_steps=N_STEPS)
    show_counterfactual_triplets(qual_batch.cpu(), activated['images_after'], f"Agnostic circuit activation: {agnostic['name']}")
    show_counterfactual_triplets(qual_batch.cpu(), suppressed['images_after'], f"Agnostic circuit suppression: {agnostic['name']}")


## 7. Summary Table

In [ ]:
summary_table = agg_df.copy()
summary_table = summary_table.sort_values(['circuit_type', 'selected_circuit', 'target_kind', 'mode'])
summary_table


In [ ]:
print('Notebook interpretation guide:')
print('- Strong target-vs-control score shifts support causal access to the discovered circuit.')
print('- For class-specific circuits, positive associated-logit / probability shifts are the main task-level evidence.')
print('- For class-agnostic circuits, entropy and confidence shifts test whether the circuit shapes evidence concentration rather than a single class.')
